# CoreML Image Classifier Demo

This notebook demonstrates the end-to-end workflow for training a sentiment model and using it for recommendations.

In [1]:
import os
import sys
import pandas as pd

# Ensure the src directory is in the path
sys.path.append(os.path.abspath('.'))

from src.sentiment_analyzer import SentimentAnalyzer
from src.content_recommender import ContentRecommender

## Configuration

In [2]:
REVIEWS_PATH = os.path.join("data", "sample_reviews.csv")
ITEMS_PATH = os.path.join("data", "sample_items.csv")
MODEL_PATH = os.path.join("models", "sentiment_model.joblib")

## Step 1: Training Sentiment Model

In [3]:
print("=== Step 1: Training sentiment model ===")
df = pd.read_csv(REVIEWS_PATH)
print("Loaded", len(df), "reviews from", REVIEWS_PATH)

analyzer = SentimentAnalyzer()
results = analyzer.train(df["text"].tolist(), df["label"].tolist())

print("Test accuracy:", round(results["accuracy"], 3))
print("Classification report:\n", results["report"])

analyzer.save(MODEL_PATH)
print("Saved model to", MODEL_PATH)

=== Step 1: Training sentiment model ===
Loaded 24 reviews from data/sample_reviews.csv
Test accuracy: 0.4
Classification report:
               precision    recall  f1-score   support

    negative       0.00      0.00      0.00         3
    positive       0.40      1.00      0.57         2

    accuracy                           0.40         5
   macro avg       0.20      0.50      0.29         5
weighted avg       0.16      0.40      0.23         5

Saved model to models/sentiment_model.joblib


## Step 2: Scoring Items

In [4]:
print("\n=== Step 2: Scoring items with sentiment model ===")
items = pd.read_csv(ITEMS_PATH)
print("Loaded", len(items), "items from", ITEMS_PATH)

sentiment_scores = []
for review in items["review"].tolist():
    score = analyzer.predict_score(review)
    sentiment_scores.append(score)

items["sentiment_score"] = sentiment_scores
print(items[["item_id", "sentiment_score"]].to_string(index=False))


=== Step 2: Scoring items with sentiment model ===
Loaded 10 items from data/sample_items.csv
item_id  sentiment_score
item_01         0.576636
item_02         0.577881
item_03         0.488979
item_04         0.598533
item_05         0.577881
item_06         0.525277
item_07         0.554736
item_08         0.488979
item_09         0.488979
item_10         0.522360


## Step 3: Fitting Recommender

In [5]:
print("\n=== Step 3: Fitting recommender ===")
recommender = ContentRecommender(sentiment_weight=0.3)
recommender.fit(
    item_ids=items["item_id"].tolist(),
    descriptions=items["description"].tolist(),
    sentiment_scores=sentiment_scores,
)


=== Step 3: Fitting recommender ===


## Step 4: Show Recommendations

In [6]:
def show_recommendations(recommender, query_id, top_k=3):
    print(f"\n=== Recommendations for {query_id} ===")
    results = recommender.recommend(query_id, top_k=top_k)
    for r in results:
        print(
            "  ->", r["item_id"],
            "| score:", r["score"],
            "| similarity:", r["similarity"],
        )

show_recommendations(recommender, "item_01", top_k=3)
show_recommendations(recommender, "item_04", top_k=3)
show_recommendations(recommender, "item_06", top_k=3)


=== Recommendations for item_01 ===
  -> item_06 | score: 0.3261 | similarity: 0.2407
  -> item_02 | score: 0.2777 | similarity: 0.1491
  -> item_04 | score: 0.1796 | similarity: 0.0

=== Recommendations for item_04 ===
  -> item_05 | score: 0.346 | similarity: 0.2466
  -> item_02 | score: 0.1734 | similarity: 0.0
  -> item_01 | score: 0.173 | similarity: 0.0

=== Recommendations for item_06 ===
  -> item_01 | score: 0.3415 | similarity: 0.2407
  -> item_07 | score: 0.2516 | similarity: 0.1217
  -> item_04 | score: 0.1796 | similarity: 0.0
